# 12 · Dataset 与 DataLoader

> **本节属于 Part 5 · 优化与训练工程。**

nb10 的训练循环里，我们手动 `np.random.shuffle(idx)`、手动 `for i in range(0, N, bs)` 切批次。本节把这些封装成 `Dataset / DataLoader`，再配合上一节的 `optim.Adam`，把训练循环写得和 PyTorch 一样清爽。

## 学习目标

- 理解 `Dataset`（怎么取一条/一批数据）与 `DataLoader`（打乱 + 分批）的职责划分
- 用 `DataLoader + optim.Adam` 重写 MNIST 训练，对比 nb10 的手写循环
- 与 PyTorch 的 `Dataset / DataLoader` 对照

## Dataset / DataLoader 的实现

`Dataset` 负责"按索引取数据"，`DataLoader` 负责"每轮打乱、按 batch 吐出"。看实现：

In [ ]:
import inspect
import numpy as np
import minitorch
from minitorch import Tensor, nn, no_grad, data
from minitorch.optim import Adam

print(inspect.getsource(data.DataLoader.__iter__))

In [ ]:
# 小演示：把数组包成数据集，用 DataLoader 分批
X = np.arange(20).reshape(10, 2); y = np.arange(10)
ds = data.TensorDataset(X, y)
dl = data.DataLoader(ds, batch_size=3, shuffle=True)
print("共", len(dl), "个 batch：")
for xb, yb in dl:
    print("  batch 标签:", yb)

## 用 DataLoader + Adam 重训 MNIST

对比一下：训练循环现在只剩"**取 batch → 前向 → 反向 → step**"，干净利落。优化器换成 Adam，收敛更快。

In [ ]:
(X_tr, y_tr), (X_te, y_te) = minitorch.utils.load_mnist(n_train=20000, n_test=5000)
train_loader = data.DataLoader(data.TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

minitorch.set_seed(0)
model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
opt = Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def accuracy():
    with no_grad():
        return (model(Tensor(X_te)).data.argmax(1) == y_te).mean()

import time
t0 = time.time()
for ep in range(8):
    for xb, yb in train_loader:               # DataLoader 负责打乱与分批
        opt.zero_grad()
        loss_fn(model(Tensor(xb)), yb).backward()
        opt.step()
    print(f"epoch {ep}  test_acc {accuracy()*100:.2f}%")
print(f"用时 {time.time()-t0:.1f}s")

> 对比 nb10 用朴素 SGD 跑 15 轮才到 95.7%，这里 Adam + DataLoader 只需几轮就更高——优化器与工程封装的价值立竿见影。

## PyTorch 对照

API 几乎一模一样：

In [ ]:
import torch
from torch.utils.data import TensorDataset as TorchDS, DataLoader as TorchDL

tds = TorchDS(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr))
tdl = TorchDL(tds, batch_size=64, shuffle=True)
print("minitorch:  data.DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)")
print("PyTorch  :  DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)")
print("一个 batch 形状:", next(iter(tdl))[0].shape)

## 📦 沉淀进 minitorch

`Dataset / TensorDataset / DataLoader` 在 **`minitorch/data/`**，由 `tests/test_layers.py` 覆盖。

## 小练习

1. **不打乱会怎样**：把 `shuffle=False` 跑一遍，准确率/收敛有变化吗？为什么打乱通常更好？
2. **batch size 的影响**：试 `batch_size=8` 与 `batch_size=512`，比较每轮耗时与收敛速度。
3. **自定义 Dataset**：写一个 `Dataset` 子类，在 `__getitem__` 里对图像做随机水平翻转（数据增强），观察对泛化的影响。

## 小结 & 下一站

✅ 有了 `DataLoader + optim`，训练循环已经和 PyTorch 一样简洁。我们的 minitorch 越来越像一个"真"框架了。

**下一站 → `13_regularization_dropout_l2`**：当模型开始**过拟合**（背下训练集却认不出新样本）时怎么办？我们将实现 **Dropout** 与 **L2 正则**，并亲眼看到它们如何改善泛化。